<a href="https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main-script-attention-head.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters
# Parallel process on GPU
batch_size = 32
# Maximum context length for predicting next token
block_size = 8
# Iterations for training the model
max_iters = 5000
# Evaluate the loss while training after regular interval
eval_interval = 300
# Learning rate: controls how much weights change at every update step
learning_rate = 1e-3
# Use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# The embedding dimension of each vocabulary.
token_embedding_size = 32
eval_iters = 200

# To maintain reproducability of random indices.
torch.manual_seed(1337)

# Download the dataset to train on.
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# All the unique characters that occur in the text
unique_characters = sorted(list(set(text)))
vocab_size = len(unique_characters)

# Mapping for each unique character.
stoi = { ch:i for i,ch in enumerate(unique_characters) }
itos = { i:ch for i,ch in enumerate(unique_characters) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

encoded_text = encode(text)

dataset = torch.tensor(encoded_text, dtype=torch.long)

# Split the data into train and validation sets
train_dataset_proportion = 0.9
number_of_dataset = int(train_dataset_proportion*len(dataset))

train_dataset = dataset[:number_of_dataset]
validation_dataset = dataset[number_of_dataset:]

def get_batch(split):
    dataset = train_dataset if split == 'train' else validation_dataset
    # Returns "batch_size" (4) random starting indices from the dataset.
    # The upper bound is "len(dataset) - block_size" (1003854 - 8) so that there are enough tokens for the block size even if the largest possible index is picked.
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    # Get "block_size" tokens starting at each chosen index, for all indices.
    context = torch.stack([dataset[i:i+block_size] for i in ix])

    # Get "block_size" tokens starting one position after each chosen index, for all indices.
    target = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    return context.to(device), target.to(device)

@torch.no_grad()
def estimate_loss():
    average_losses = {}

    # Switch to eval mode. Layers like dropout behave differently during evaluation.
    # In eval model, the dropout layer does not drop some neurons on every forward pass.
    # For the current bigram model, the following code has no effect, but needed once dropout is added later.
    model.eval()

    # Run the same evaluation twice, once on training data and once on validation data.
    # Comparing the two tells you if the model is memorizing when loss calculating using training data is much lower than the one calculating using validation data).
    for split in ['train', 'val']:
        # Creating a tensor with eval_iters (200) number of values to store the value of loss.
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            # Get a fresh random batch from the current split of the shape (batch_size, block_size)
            # Example: context_ids shape (32, 8), target_ids shape (32, 8)
            # context_ids tensor([[39, 52, 41, 39, 11,  0, 13, 52],  <==  0
            #                     [ 1, 47, 58,  1, 52, 53, 58,  1],  <==  1
            #                     [58, 46, 43,  0, 40, 53, 58, 57],  <==  2
            #                     [ 6,  0, 18, 43, 39, 57, 58,  1],  <==  3
            #                     [47, 52, 45,  2,  1, 20, 39, 57],  <==  4
            #                     [39,  1, 40, 56, 47, 43, 44, 43],  <==  5
            #                     [37, 53, 59,  1, 61, 43, 56, 43],  <==  6
            #                     [53, 51, 43,  1, 61, 53, 52, 42],  <==  7
            #                     [39, 56, 49,  1, 51, 43,  6,  1],  <==  8
            #                     [60, 43,  1, 57, 58, 59, 44, 44],  <==  9
            #                     [ 1, 51, 43,  1, 58, 46, 63,  1],  <== 10
            #                     [ 1, 56, 47, 45, 46, 58,  1, 34],  <== 11
            #                     [40, 59, 52, 42, 39, 52, 41, 43],  <== 12
            #                     [43, 56, 43, 12,  1, 31, 46, 39],  <== 13
            #                     [46, 59, 51, 39, 52,  1, 41, 39],  <== 14
            #                     [45, 56, 39, 57, 57,  1, 50, 53],  <== 15
            #                     [ 1, 57, 53, 52,  8,  0, 13, 52],  <== 16
            #                     [ 8,  0, 31, 53,  1, 57, 46, 39],  <== 17
            #                     [53,  1, 58, 46, 43,  1, 43, 52],  <== 18
            #                     [53, 51,  1, 51, 63,  1, 51, 53],  <== 19
            #                     [50, 42,  0, 48, 43, 56, 49, 47],  <== 20
            #                     [43, 58, 41, 46,  1, 42, 43, 61],  <== 21
            #                     [40, 56, 39, 60, 43,  1, 51, 43],  <== 22
            #                     [ 1, 50, 47, 44, 58,  0, 58, 46],  <== 23
            #                     [25, 21, 30, 13, 26, 16, 13, 10],  <== 24
            #                     [12,  1, 52, 39, 63,  6,  1, 41],  <== 25
            #                     [63, 53, 59,  1, 45, 56, 53, 61],  <== 26
            #                     [43, 58, 56, 59, 41, 46, 47, 53],  <== 27
            #                     [47, 52, 42,  6,  1, 63, 53, 59],  <== 28
            #                     [53, 58, 46, 43, 56,  1, 56, 43],  <== 29
            #                     [33, 15, 20, 21, 27, 10,  0, 35],  <== 30
            #                     [10,  0, 32, 59, 52, 47, 57,  1]]) <== 31
            #
            #                       0   1   2   3   4   5   6   7
            context_ids, target_ids = get_batch(split)

            # Forward pass. Returns logits and the average loss over all predictions in this batch.
            logits, loss = model(context_ids, target_ids)

            # loss is a tensor. .item() pulls out the plain Python number and stores it at position k.
            # After k=0: losses = [4.31, 0.,   0.,   0.,   0.  ]
            # After k=1: losses = [4.31, 4.19, 0.,   0.,   0.  ]
            # After k=4: losses = [4.31, 4.19, 4.25, 4.40, 4.22]
            losses[k] = loss.item()

        # Average all the batch losses into one number and store it under this split's name.
        # Example: (4.31 + 4.19 + 4.25 + 4.40 + 4.22) / 5 = 4.274
        # out = {'train': 4.274}
        average_losses[split] = losses.mean()

    # In training mode, it randomly turns off some neurons on every forward pass. This stops model from depending too heavily on any single neuron.
    # The model is forced to learn backup paths, so it generalizes better instead of memorizing the training data.
    # No effect on the current bigram model, but needed once dropout is added later.
    model.train()

    return average_losses

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()

        # Each token can also be represented by three values: Query, Key and Value.
        # These values are embeddings but different from a token embedding.

        # Key represents the side of the token that asks what information does this token offer to others.
        #
        # Following key matrix is multiplied with each token's embedding to get the key vector of that token.
        # Hence, the Batch Size and Block Size do not matter for key matix. Only the token's embedding size does.
        #
        # A token's embedding is 1 X token_embedding_size, so for the multiplication to work the key matrix must have token_embedding_size rows.
        # The number of columns is head_size, and it decides the size of the resulting key vector.
        # So the size of a token's key vector is not the same as the size of its embedding.
        # head_size is usually calculated by dividing token_embedding_size by the number of heads in each layer.
        #
        # Initially the values of these weights are random and they get updated during training.
        #
        # self.key.weight:
        #
        #  [[    0.60,   -0.30 ],
        #   [   -0.20,    0.50 ],
        #   [    0.40,    0.10 ]]
        #
        self.key = nn.Linear(token_embedding_size, head_size, bias=False)

        # Query represents the side of the token that asks what information is this token looking for.
        #
        # [Same concept as K]
        self.query = nn.Linear(token_embedding_size, head_size, bias=False)

        # Value represents the side of the token that presents the value that this token adds.
        #
        # [Same concept as K]
        self.value = nn.Linear(token_embedding_size, head_size, bias=False)

        # Creates a square matrix of size block_size x block_size, filled with 1s.
        #
        # With block_size = 4:
        #
        #   [ 1, 1, 1, 1 ]
        #   [ 1, 1, 1, 1 ]
        #   [ 1, 1, 1, 1 ]
        #   [ 1, 1, 1, 1 ]
        ones_matrix = torch.ones(block_size, block_size)

        # Modify the matrix to make it lower triangular using 1s and all the other item becomes 0.
        #
        # This is needed because the model is trained to predict the next token at every position simultaneously.
        # Position 2 is being trained to predict token 3, so while building the attention aware embedding for position 2, it must only look at positions 0, 1 and 2.
        # If it could look at position 3, it would already see the answer it is supposed to predict.
        #
        # With block_size = 4:
        #
        #            Pos 0  Pos 1  Pos 2  Pos 3
        #   Pos 0  [     1,     0,     0,     0 ]   <== Position 0 sees only itself
        #   Pos 1  [     1,     1,     0,     0 ]   <== Position 1 sees positions 0 and 1
        #   Pos 2  [     1,     1,     1,     0 ]   <== Position 2 sees positions 0, 1, 2
        #   Pos 3  [     1,     1,     1,     1 ]   <== Position 3 sees everything
        causal_mask = torch.tril(ones_matrix)

        # Declare self.tril, holding the value of causal_mask. Same as,
        #
        # self.tril = causal_mask
        #
        # register_buffer is used because it adds two things.
        # First, when the model is moved to the GPU with model.to(device), the mask moves with it.
        # A plain attribute would stay on the CPU and the code would crash when it meets the GPU tensors.
        # Second, the mask gets saved and loaded along with the model.
        #
        # register_buffer is used rather than nn.Parameter because a parameter is trainable and the optimizer updates it every step.
        # There is nothing to learn here. The mask is the same numbers before and after training.
        self.register_buffer('tril', causal_mask)

    def forward(self, x):
        #  x = Token Embedding + Position Embedding

        # B = 2, T = 3, C = 3
        #          [
        # Batch 0:  [
        #            [ 0.41,  0.12, -0.57],
        #            [ 0.24,  0.65, -0.24],
        #            [ 0.17, -0.12,  0.39]
        #           ],
        # Batch 1:  [
        #            [-0.39,  0.52,  0.13],
        #            [ 0.44,  0.15, -0.54],
        #            [ 0.77, -0.02,  0.49]
        #           ]
        #          ]
        B,T,C = x.shape

        # Multiplies each token's embedding with the key matrix, to get the key vector of that token.
        #
        # Batch 0, position 0. Embedding [0.41, 0.12, -0.57]:
        #   col 0:  0.41 * 0.60  +  0.12 * (-0.20)  +  (-0.57) * 0.40  = 0.246 - 0.024 - 0.228 = -0.006
        #   col 1:  0.41 * (-0.30)  +  0.12 * 0.50  +  (-0.57) * 0.10  = -0.123 + 0.060 - 0.057 = -0.120
        #   key = [-0.006, -0.120]
        #
        # Batch 0, position 1. Embedding [0.24, 0.65, -0.24]:
        #   col 0:  0.24 * 0.60  +  0.65 * (-0.20)  +  (-0.24) * 0.40  = 0.144 - 0.130 - 0.096 = -0.082
        #   col 1:  0.24 * (-0.30)  +  0.65 * 0.50  +  (-0.24) * 0.10  = -0.072 + 0.325 - 0.024 =  0.229
        #   key = [-0.082, 0.229]
        #
        # The same runs for all the tokens in every batch.
        #
        # k
        #          [
        # Batch 0:  [
        #            [-0.006, -0.120],
        #            [-0.082,  0.229],
        #            [ 0.282, -0.072]
        #           ],
        # Batch 1:  [
        #            [-0.286,  0.390],
        #            [ 0.018, -0.111],
        #            [ 0.662, -0.192]
        #           ]
        #          ]
        #
        # Shape goes from (B, T, C) to (B, T, head_size).
        k = self.key(x)

        # [Same concept as k]
        q = self.query(x)

        # [Same concept as k]
        v = self.value(x)

        # The transpose is needed because a matrix multiply pairs the columns of the left matrix with the rows of the right one.
        # q is (T, hs), so its columns are the hs dimension. k is also (T, hs), so its rows are the T dimension.
        # Those do not match, and the multiply would fail.
        # Flipping k puts its hs dimension on the rows, so hs now meets hs and the dot product runs over the key vector.
        #
        # k, batch 0. Shape (T=3, hs=2):
        #
        #            hs0     hs1
        #   pos 0 [ -0.006, -0.120 ]
        #   pos 1 [ -0.082,  0.229 ]
        #   pos 2 [  0.282, -0.072 ]
        #
        # k.transpose(-2,-1), batch 0. Shape (hs=2, T=3):
        #
        #            pos0    pos1    pos2
        #   hs 0 [ -0.006, -0.082,  0.282 ]
        #   hs 1 [ -0.120,  0.229, -0.072 ]
        k_transpose = k.transpose(-2,-1)

        # We will be calculating the dot product of k and q.
        # A dot product adds up that many multiplications as the size of k or q, so the more numbers there are, the larger the scores get.
        #
        # With head_size = 2, one score is the sum of 2 multiplications:
        #   q[0] = [-0.152, 0.539],  k[1] = [-0.082, 0.229]
        #   (-0.152 * -0.082) + (0.539 * 0.229) = 0.0125 + 0.1234 = 0.1359
        #
        # With head_size = 64, the same operation sums 64 multiplications, so the total lands much higher.
        #
        # Dividing by the square root of head_size cancels that growth and keeps the scores in a workable range.
        #
        # This formula returns sqrt(head_size).
        square_root_of_head_size = k.shape[-1]**0.5

        # We calculate the dot product of every token's q and k.
        # A high score means that query and that key point in a similar direction, so that token is relevant.
        #
        # This is also what produces a score for every pair of positions.
        # (B, T, hs) @ (B, hs, T) -> (B, T, T), so T queries meet T keys and each row of the result holds one query's scores against all keys.
        #
        # Every score is divide by square_root_of_head_size.
        #
        # wei, shape (B=2, T=3, T=3). Each item in the matrix stores the qk score with each token.
        #
        # Batch 0:        pos0     pos1     pos2
        #     pos 0  [ -0.0451,  0.0961, -0.0578 ]
        #     pos 1  [ -0.0344,  0.0505,  0.0256 ]
        #     pos 2  [  0.0050, -0.0122,  0.0111 ]
        wei = q @ k_transpose / square_root_of_head_size

        # self.tril is (block_size, block_size), so it is sliced to [:T, :T] to match the current context length.
        #
        # During generation the context starts at length 1 and grows, so it is often shorter than block_size.
        #
        #            pos0  pos1  pos2
        #   pos 0  [    1,    0,    0 ]
        #   pos 1  [    1,    1,    0 ]
        #   pos 2  [    1,    1,    1 ]
        #
        # The "== 0" changes all the items in the matrix from 0 or 1 to boolean.
        # True marks the positions the token is not allowed to look at.
        #
        #            pos0    pos1    pos2
        #   pos 0  [ False,  True,   True ]
        #   pos 1  [ False, False,   True ]
        #   pos 2  [ False, False,  False ]
        #
        blocked_positions = self.tril[:T, :T] == 0

        # blocked_positions is (T, T) and wei is (B, T, T), so the same mask is used for every batch item.
        # Wherever the mask is True, that score in wei is replaced with -inf. The rest keep their score.
        # -inf is used so that softmax turns it into exactly 0.
        # As a result, those positions end up with no attention at all.
        #
        # Batch 0:      pos0     pos1     pos2
        #     pos 0  [ -0.0451,    -inf,    -inf ]   <== position 0 sees only itself
        #     pos 1  [ -0.0344,  0.0505,    -inf ]   <== position 1 sees positions 0 and 1
        #     pos 2  [  0.0050, -0.0122,  0.0111 ]   <== position 2 sees everything
        wei = wei.masked_fill(blocked_positions, float('-inf'))

        # Turns the raw scores into proportions.
        # It first runs every score through e^x, which makes all of them positive, then divides each by the total of the row.
        # So each row now says what share of its attention this position gives to each other position.
        # dim=-1 runs it along the last dimension, so each position's scores are turned into proportions on their own.
        #
        # Batch 0:      pos0     pos1     pos2
        #     pos 0  [  1.0000,  0.0000,  0.0000 ] <== all attention on itself, nothing else is available
        #     pos 1  [  0.4788,  0.5212,  0.0000 ]
        #     pos 2  [  0.3346,  0.3288,  0.3366 ]
        wei = F.softmax(wei, dim=-1)

        # Each row of wei adds up to 1, so the scores are proportions.
        # Each score is multiplied with the value vector of the token at that column, so a score of 0.5 takes half of that token's value.
        # Adding those up gives one output vector per position, made from the tokens it attended to.
        #
        # v, batch 0:  [[ 0.325, -0.133],  <== v for first token
        #               [ 0.510, -0.089],  <== v for second token
        #               [-0.155,  0.275]]  <== v for third token
        #
        # Position 1 takes 0.479 of position 0's value plus 0.521 of its own:
        #   0.4788 * [ 0.325, -0.133] + 0.5212 * [ 0.510, -0.089] = [0.4214, -0.1101]
        #
        # out, shape (B=2, T=3, hs=2):
        #
        # Batch 0:  [[ 0.3250, -0.1330],   <== position 0, just its own value
        #            [ 0.4214, -0.1101],   <== position 1, blend of positions 0 and 1
        #            [ 0.2243,  0.0188]]   <== position 2, blend of all three
        #
        # This is the attention aware embedding. Each token now carries information from the tokens before it.
        out = wei @ v

        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, number_of_heads, head_size):
        super().__init__()

        # Initiate number_of_heads Head object.
        list_of_heads = [Head(head_size) for _ in range(number_of_heads)]

        # nn.ModuleList registers every head with the model, so their weights show up in m.parameters() and get trained.
        self.heads = nn.ModuleList(list_of_heads)

    def forward(self, x):
        # Every head reads the whole x.
        # Each head returns a shorter array of length head_size.
        #
        # x: [ 0.41, 0.12, -0.57, 0.24, 0.65, -0.24, 0.17, -0.12 ]   <== 8 numbers
        #
        # Head(x)   <== Head 0   |________ number_of_heads = 2
        # Head(x)   <== Head 1   |
        #
        #
        #                          head_size = 4
        #                    ____________|____________
        # head_outputs[0]: [ 0.32, -0.13, 0.51, -0.09 ]   <== 4 numbers
        # head_outputs[1]: [ 0.14,  0.61, 0.20,  0.48 ]   <== 4 numbers
        head_outputs = [h(x) for h in self.heads]

        # Concatenates the outputs of all the heads.
        #
        #       [ 0.32, -0.13, 0.51, -0.09 ] + [ 0.14, 0.61, 0.20, 0.48 ]
        #
        #   ->  [ 0.32, -0.13, 0.51, -0.09, 0.14, 0.61, 0.20, 0.48 ]
        #         \________head 0________/  \________head 1_______/
        #
        # With this, the length of final output becomes equal to token_embedding_size (the length of x).
        # This is exactly why head_size is set to token_embedding_size / number_of_heads, so 2 heads of 4 add back up to 8.
        return torch.cat(head_outputs, dim=-1)

class FeedForward(nn.Module):
    def __init__(self, token_embedding_size):
        super().__init__()

        # Attention lets every token collect information from the previous tokens, but it stops there.
        # All it has is a weighted average of the value vectors of previous tokens.
        # This layer is where each token processes what it collected, on its own, without looking at any other token.

        # nn.Sequential holds the layers in order and runs them one after another when it is called.
        self.net = nn.Sequential(
            # nn.Linear holds a weight matrix of shape (token_embedding_size, token_embedding_size), plus a bias vector.
            # The weights start random and get updated during training.
            #
            # With token_embedding_size = 4:
            #
            #   [  0.50, -0.20,  0.30,  0.10 ]
            #   [  0.40,  0.60, -0.50,  0.20 ]
            #   [ -0.30,  0.10,  0.70, -0.40 ]
            #   [  0.20, -0.60,  0.10,  0.50 ]
            nn.Linear(token_embedding_size, token_embedding_size),

            # ReLU takes each number on its own and replaces it with 0 if it is negative, otherwise leaves it as it is.
            nn.ReLU()
        )

    def forward(self, x):
        # x holds a context aware embedding for each of the T tokens, so its shape is (B, T, token_embedding_size).
        #
        # Following one token, [0.32, -0.13, 0.51, -0.09]:
        #
        # Step 1, the Linear. Each token is mutiplied with the matrix in Linear and bias is added to the result.
        #   col 0:  0.32 * 0.50  + (-0.13) * 0.40  + 0.51 * (-0.30) + (-0.09) * 0.20  = -0.063
        #   col 1:  0.32 * (-0.20) + (-0.13) * 0.60  + 0.51 * 0.10  + (-0.09) * (-0.60) = -0.037
        #   col 2:  0.32 * 0.30  + (-0.13) * (-0.50) + 0.51 * 0.70  + (-0.09) * 0.10  =  0.509
        #   col 3:  0.32 * 0.10  + (-0.13) * 0.20  + 0.51 * (-0.40) + (-0.09) * 0.50  = -0.243
        #
        #   result: [-0.063, -0.037, 0.509, -0.243]
        #
        # Step 2, the ReLU. Negatives become 0.
        #
        #   result: [ 0.000,  0.000, 0.509,  0.000]
        return self.net(x)

class Block(nn.Module):
    def __init__(self, token_embedding_size, number_of_heads):
        super().__init__()

        head_size = token_embedding_size // number_of_heads

        self.self_attention_head = MultiHeadAttention(number_of_heads, head_size)

        self.feedforward = FeedForward(token_embedding_size)

    def forward(self, x):
        # Add previous token's value based on their attention scores to make the embedding context aware.
        x = self.self_attention_head(x)

        # Each token processes the context aware embedding it got from attention, on its own.
        x = self.feedforward(x)

        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # A lookup table mapping each token id to a row of token_embedding_size (32 columns) numbers. Each cell is assigned random values before training.
        # Each token id is mapped to the row that contains the embedding (carrying semantic meaning) of the token.
        self.token_embedding_table = nn.Embedding(vocab_size, token_embedding_size)

        # The embeddings are then converted into logits that are used to predict the next token.
        # This layer of weights takes a row of token_embedding_size numbers and produces vocab_size logits, one score per token in the vocabulary.
        self.language_model_head = nn.Linear(token_embedding_size, vocab_size)

        # A lookup table mapping each token's position in the context ids to a row of token_embedding_size numbers. Each cell is assigned random values before training.
        # Each token's position is mapped to the row that carries the embedding for the token position.
        self.position_embedding_table = nn.Embedding(block_size, token_embedding_size)

        self.blocks = nn.Sequential(
            Block(token_embedding_size, 4),
            Block(token_embedding_size, 4),
            Block(token_embedding_size, 4),
        )

    def forward(self, context_ids, targets=None):
        # Context Ids  (B, T)
        # [[5, 2, 0],     <== Batch 0
        #  [1, 5, 3]]     <== Batch 1

        # B = 2, T = 3
        B, T = context_ids.shape

        # Gets the context batch matrix and returns a token embedding array (token_embedding_size size) for each index in the matrix.
        #
        #
        # Token Embedding Table
        #
        # row 0  [ 0.10, -0.20,  0.30]
        # row 1  [-0.40,  0.50,  0.10]
        # row 2  [ 0.20,  0.60, -0.30]
        # row 3  [ 0.70, -0.10,  0.40]
        # row 4  [-0.50,  0.30,  0.20]
        # row 5  [ 0.40,  0.10, -0.60]
        #
        #
        # Token Embedding (B, T, C)
        #          [
        # Batch 0:  [
        #            [ 0.40,  0.10, -0.60],   <== row 5
        #            [ 0.20,  0.60, -0.30],   <== row 2
        #            [ 0.10, -0.20,  0.30]    <== row 0
        #           ],
        # Batch 1:  [
        #            [-0.40,  0.50,  0.10],   <== row 1
        #            [ 0.40,  0.10, -0.60],   <== row 5
        #            [ 0.70, -0.10,  0.40]    <== row 3
        #           ]
        #          ]
        token_embedding = self.token_embedding_table(context_ids) # Token Embedding's Shape = Batch Size: B, Block Size: T, Token Embedding Size: C)

        # [0, 1, 2]
        position_ids = torch.arange(T, device=device)

        # Position Embedding Table
        #
        # Pos 0  [ 0.01,  0.02,  0.03]
        # Pos 1  [ 0.04,  0.05,  0.06]
        # Pos 2  [ 0.07,  0.08,  0.09]
        #
        #
        # Position Embedding (T, C)
        # [[ 0.01,  0.02,  0.03],
        #  [ 0.04,  0.05,  0.06],
        #  [ 0.07,  0.08,  0.09]]
        position_embedding = self.position_embedding_table(position_ids)

        # x (B, T, C)
        #          [
        # Batch 0:  [
        #            [ 0.41,  0.12, -0.57],   <== 0.40+0.01, 0.10+0.02, -0.60+0.03
        #            [ 0.24,  0.65, -0.24],
        #            [ 0.17, -0.12,  0.39]
        #           ],
        # Batch 1:  [
        #            [-0.39,  0.52,  0.13],
        #            [ 0.44,  0.15, -0.54],
        #            [ 0.77, -0.02,  0.49]
        #           ]
        #          ]
        x = token_embedding + position_embedding

        x = self.blocks(x)

        # Gets the token embedding array for and returns a logits array (vocab_size size) for each index in the matrix.
        #
        # Logits (B, T, C)
        #          [
        # Batch 0:  [
        #            [ 0.12, -0.31,  0.45,  0.08, -0.22,  0.19],   <== scores for the token after position 0
        #            [-0.05,  0.27, -0.14,  0.33,  0.11, -0.40],
        #            [ 0.21,  0.02,  0.38, -0.17,  0.29,  0.06]
        #           ],
        # Batch 1:  [
        #            [ 0.09,  0.41, -0.23,  0.15, -0.08,  0.30],
        #            [ 0.34, -0.12,  0.07,  0.22, -0.35,  0.18],
        #            [-0.19,  0.25,  0.31,  0.04,  0.13, -0.27]
        #           ]
        logits = self.language_model_head(x) # Logit's Shape = (Batch Size: B, Block Size: T, Vocab Size: C)

        if targets is None:
            return logits, None

        # Calculate the loss, based on the targets.

        B, T, C = logits.shape

        # Flattens the first two dimensions into one, so the shape goes from (Batch Size, Block Size, Vocab Size) to (Batch Size * Block Size, Vocab Size).
        #               Batch Item 1      Batch Item 2      Batch Item 3      Batch Item 4
        #               ________________  ________________  ________________  _________________
        # For example, [[[1, 2], [2, 3]], [[3, 4], [4, 5]], [[5, 6], [6, 7]], [[7, 8], [8, 9]]] (three nested arrays) becomes
        #               ______________  ______________  ______________  ______________
        #              [[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]] (two nested arrays).
        # Essentially, the rows of all the batch items are joined into one flat list.
        # This is needed because the loss function expects the logits in the shape (Predictions, Vocab Size).
        logits = logits.view(B*T, C)

        # Converting the target matrix also into the same dimension as logits.
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, context_ids, max_new_tokens):
        # context_ids' shape = (B, T). Array of B Batch Items. Each item has T context ids.
        # context_ids = [[5, 2, 8], <== Batch 1
        #                [7, 1, 9]] <== Batch 2
        #                 ^  ^  ^
        #                 |  |  |
        #                 T  T  T
        #                 O  O  O
        #                 K  K  K
        #                 E  E  E
        #                 N  N  N
        #
        #                 1  2  3
        #
        # Shape: (2, 3)
        for _ in range(max_new_tokens):
            # Crop the context ids to the last block_size tokens.
            context_ids_cropped = context_ids[:, -block_size:]

            logits, _ = self(context_ids_cropped)

            # Extract logits of the last token index for each batch.
            # Example: [[0.1, -0.5,  0.3, ..., 0.2],  <== Batch 1; Logits for token id index 8
            #           [0.2,  0.1, -0.1, ..., 0.4]]  <== Batch 2; Logits for token id index 9
            last_token_index_logits = logits[:, -1, :]

            # Get the probabilities by applying softmax to all the logits of the last token index.
            # Example: [[0.02, 0.01, 0.05, ..., 0.03], <== Batch 1; Probabilities for each token id index, after token id index 8
            #           [0.03, 0.02, 0.01, ..., 0.04]] <== Batch 2; Probabilities for each token id index, after token id index 9
            probabilities = F.softmax(last_token_index_logits, dim=-1)

            # Returns indices of the next predicted token, for all the batches.
            # Example: [[42], <== Batch 1: Next token id index, after index 8
            #           [15]] <== Batch 2: Next token id index, after index 9
            indices_of_next_token = torch.multinomial(
                probabilities,
                # Randomly returns 1 token id index per batch based on the probability distribution.
                num_samples=1
            )

            # Append the predicted index for each batch at the end.
            # Updated: [[5, 2, 8, 42],
            #           [7, 1, 9, 15]]
            context_ids = torch.cat((context_ids, indices_of_next_token), dim=1)
        return context_ids

model = BigramLanguageModel()
m = model.to(device)

# Create an AdamW optimizer object (a specific optimization algorithm).
optimizer = torch.optim.AdamW(
                # Pass all model parameters (token_embedding_table which contains weights) to be updated during training.
                m.parameters(),
                # Learning rate: controls how much weights change at every update step.
                # Lower lr means slower but more stable learning while, Higher lr means faster but may diverge.
                # new_weight = old_weight - (learning_rate * gradient)
                lr=learning_rate
            )

for steps in range(max_iters):
    # Every once in a while evaluate the loss on train and validation sets
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    context_ids, target_ids = get_batch('train')
    logits, loss = m(context_ids, target_ids)

    # Each cell in the token embedding table contains weights and each weights also has the property gradient.
    # The gradient is the derivative of loss with respect of weight. Hence, it indicates how much does the loss change by a minor change in weights.
    # The increase and decrease in the value of each weights is decided by this gradient.

    # We need to ensure that before we start calculating and storing the values in the gradient property of these weights, all the existing gradient properties in each weight of the token embedding table are cleared.
    # In short, it should not have the values calculate in the previous iteration.
    optimizer.zero_grad(set_to_none=True)

    # The loss goes back to each weight to calculate how much did that weight contributed into the loss.
    # Based on that calculates the gradient for that weights and stored the value in the gradient property of the weight.
    loss.backward()

    # This uses the calculated gradient value and update each weight cell in the token embedding table.
    optimizer.step()

def generate_next_tokens(context_ids, max_new_tokens=100):
    # Generate next tokens using the given context ids.
    generated_tokens = m.generate(context_ids, max_new_tokens)

    # Get the tokens for the first batch.
    generated_tokens_batch_1 = generated_tokens[0]

    # Convert the tokens array from tensor to list.
    token_id_index_list = generated_tokens_batch_1.tolist()

    # Use the tokens list to decode it using tokenizer.
    generated_text = decode(token_id_index_list)

    return generated_text

initial_context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_next_tokens = generate_next_tokens(initial_context, 500)
print(generated_next_tokens)

--2026-09-09 11:28:43--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-09-09 11:28:44 (33.1 MB/s) - ‘input.txt.1’ saved [1115394/1115394]

step 0: train loss 4.2092, val loss 4.2073
step 300: train loss 3.1483, val loss 3.1665
step 600: train loss 2.8628, val loss 2.8681
step 900: train loss 2.6941, val loss 2.7089
step 1200: train loss 2.5902, val loss 2.5979
step 1500: train loss 2.5113, val loss 2.5217
step 1800: train loss 2.4788, val loss 2.4787
step 2100: train loss 2.4404, val loss 2.4431
step 2400: train loss 2.4113, val l